In [1]:
from functools import partial

import os
import xgboost as xgb
from imblearn.over_sampling import SMOTE, SVMSMOTE, BorderlineSMOTE, KMeansSMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from utils import RANDOM_SEED, RESULTS_DIR, BinaryDatasetManager, ParamRunner

In [2]:
data_manager = BinaryDatasetManager()

data_comphrasion_df = data_manager.df[
    [
        "ratio (0 : 1)",
        "shape",
        "inbalance strength",
        "class negative num",
        "class positive num",
    ]
]
data_comphrasion_df

,ratio (0 : 1),shape,inbalance strength,class negative num,class positive num
Breast Cancer Wisconsin (Diagnostic),357:212,"(569, 30)",1.68,357,212
Ionosphere,14:25,"(351, 34)",1.79,126,225
Pima Indians Diabetes,125:67,"(768, 8)",1.87,500,268
ecoli,43:5,"(336, 7)",8.60,301,35
optical_digits,2533:277,"(5620, 64)",9.14,5066,554
satimage,5809:626,"(6435, 36)",9.28,5809,626
pen_digits,9937:1055,"(10992, 16)",9.42,9937,1055
abalone,3786:391,"(4177, 10)",9.68,3786,391
sick_euthyroid,2870:293,"(3163, 42)",9.80,2870,293
spectrometer,54:5,"(531, 93)",10.80,486,45


In [3]:
data_comphrasion_df.to_latex(
    os.path.join(RESULTS_DIR, "data_comphrasion_df.tex"),
    float_format="%.2f",
    column_format="lcccc",
    escape=False,
)

# Setup

In [4]:
smote_kwargs = {
    "sampling_strategy": "minority",
    "k_neighbors": 5,
}
m_neighbors = 10

smote_class = partial(
    SMOTE,
    **smote_kwargs,
)

borderline_smote_class = partial(
    BorderlineSMOTE, **smote_kwargs, kind="borderline-1", m_neighbors=m_neighbors
)

kmeans_smote_class = partial(KMeansSMOTE, **smote_kwargs, m_neighbors=m_neighbors)

svm_smote_class = partial(SVMSMOTE, **smote_kwargs, m_neighbors=m_neighbors)

SMOTE_CLASSES = {
    "SMOTE": smote_class,
    "BorderlineSMOTE": borderline_smote_class,
    "KMeansSMOTE": kmeans_smote_class,
    "SVMSMOTE": svm_smote_class,
}

# Trees

In [5]:
tree_kwargs = {
    "criterion": "gini",
    "min_samples_split": 10,
    "min_samples_leaf": 5,
    "max_features": None,  # all features
}

tree_class = partial(
    DecisionTreeClassifier,
    **tree_kwargs,
)

random_forest_class = partial(
    RandomForestClassifier,
    n_jobs=-1,
    **tree_kwargs,
)

xgb_class = partial(
    xgb.XGBClassifier,
    n_jobs=-1,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=RANDOM_SEED,
    device="gpu",
)

ESTIMATORS = {
    "DecisionTreeClassifier": tree_class,
    "RandomForestClassifier": random_forest_class,
    "XGBClassifier": xgb_class,
}

param_grid = {
    "n_bagging_estimators": [10, 25, 50],
    "n_estimators": [10, 25, 50],
    "max_depth": [5, 10, 20],
}

In [ ]:
run_id = 0
n_takes = 5
results_dir = os.path.join(RESULTS_DIR, f"run_{run_id}")
os.makedirs(results_dir, exist_ok=True)
performed_runs = set(os.listdir(results_dir))

for run in performed_runs:
    if run.startswith("run_"):
        run_id = max(run_id, int(run.split("_")[1]))

for smote_name, smote_class in sorted(SMOTE_CLASSES.items(), key=lambda x: x[0])[:2]:
    for dataset_name, (
        (X_train, y_train),
        (_, _),
        (X_test, y_test),
    ) in data_manager.serve(oversampler=smote_class()):
        for model_name, model_class in sorted(ESTIMATORS.items(), key=lambda x: x[0]):
            for option in ("bagging", "no_bagging"):
                for take in range(1, n_takes + 1):
                    random_state = RANDOM_SEED + take

                    cur_param_grid = param_grid.copy()
                    if model_name == "DecisionTreeClassifier":
                        cur_param_grid.pop("n_estimators")
                    if option == "no_bagging":
                        cur_param_grid.pop("n_bagging_estimators")

                    filename = f"{dataset_name}__{smote_name}__{model_name}__{option}__{take}.pkl"
                    if filename in performed_runs:
                        print(
                            f"Already performed {smote_name} with {model_name} on {dataset_name} with {option} take {take}"
                        )
                        continue
                    print(
                        f"Performing {smote_name} with {model_name} on {dataset_name} with {option} take {take}"
                    )

                    runner = ParamRunner(
                        base_estimator_class=model_class,
                        param_grid=cur_param_grid,
                        scoring={
                            "accuracy": "accuracy",
                            "f1": "f1",
                            "recall": "recall",
                            "precision": "precision",
                            "roc_auc": "roc_auc",
                        },
                        option=option,
                        n_jobs=6,
                        random_state=random_state,
                    )
                    runner.fit(X_train, y_train, X_test, y_test)

                    runner.results_.to_json(
                        os.path.join(results_dir, filename),
                        orient="records",
                        lines=True,
                    )

Already performed SMOTE with DecisionTreeClassifier on Breast Cancer Wisconsin (Diagnostic) with bagging take 1
Already performed SMOTE with DecisionTreeClassifier on Breast Cancer Wisconsin (Diagnostic) with bagging take 2
Already performed SMOTE with DecisionTreeClassifier on Breast Cancer Wisconsin (Diagnostic) with bagging take 3
Already performed SMOTE with DecisionTreeClassifier on Breast Cancer Wisconsin (Diagnostic) with bagging take 4
Already performed SMOTE with DecisionTreeClassifier on Breast Cancer Wisconsin (Diagnostic) with bagging take 5
Already performed SMOTE with DecisionTreeClassifier on Breast Cancer Wisconsin (Diagnostic) with no_bagging take 1
Already performed SMOTE with DecisionTreeClassifier on Breast Cancer Wisconsin (Diagnostic) with no_bagging take 2
Already performed SMOTE with DecisionTreeClassifier on Breast Cancer Wisconsin (Diagnostic) with no_bagging take 3
Already performed SMOTE with DecisionTreeClassifier on Breast Cancer Wisconsin (Diagnostic) wit

Parameter Grid:   0%|          | 0/27 [00:00<?, ?it/s]

Performing SMOTE with RandomForestClassifier on Breast Cancer Wisconsin (Diagnostic) with bagging take 3


Parameter Grid:   0%|          | 0/27 [00:00<?, ?it/s]

Performing SMOTE with RandomForestClassifier on Breast Cancer Wisconsin (Diagnostic) with bagging take 4


...

Parameter Grid:   0%|          | 0/27 [00:00<?, ?it/s]

Performing SMOTE with RandomForestClassifier on optical_digits with bagging take 3


Parameter Grid:   0%|          | 0/27 [00:00<?, ?it/s]

Process SpawnPoolWorker-804:
Process SpawnPoolWorker-799:
Traceback (most recent call last):
  File "/opt/homebrew/Caskroom/miniconda/base/envs/automl/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/homebrew/Caskroom/miniconda/base/envs/automl/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/homebrew/Caskroom/miniconda/base/envs/automl/lib/python3.11/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/opt/homebrew/Caskroom/miniconda/base/envs/automl/lib/python3.11/multiprocessing/queues.py", line 364, in get
    with self._rlock:
  File "/opt/homebrew/Caskroom/miniconda/base/envs/automl/lib/python3.11/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
           ^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
Traceback (most recent call last):
  File "/opt/homebrew/Caskroom/miniconda/base/envs/au

KeyboardInterrupt: 